# Pandas Basics for Machine Learning

**Summer of Science 2026 – CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

## Why this notebook

Most Machine Learning work starts with understanding and cleaning data. While learning Pandas, I realized that a few core operations are used repeatedly in almost every dataset.

This notebook is a personal reference for the Pandas functions and workflows that I found most useful and expect to reuse in future projects.

## Contents
1. Creating DataFrames and understanding the structure
2. `info()` and `describe()` — the first two commands to run on any dataset
3. Selecting data: `[]`, `.loc[]`, `.iloc[]`
4. Filtering rows with conditions
5. Handling missing values in Pandas
6. Adding and removing columns
7. `groupby()` and `value_counts()` for categorical analysis
8. Sorting and ranking
9. Converting to NumPy — the handoff to sklearn
10. Mini-exercise: Building a data summary report function

In [1]:
import pandas as pd
import numpy as np
print('Pandas version:', pd.__version__)

Pandas version: 2.3.3


## Creating DataFrames

In [2]:
# Creating from a dictionary — the most common way when building test data
data = {
    'age':          [25, 34, 45, 28, 52, 31, 38, None],
    'income':       [50000, 82000, 120000, 45000, 95000, 60000, 74000, 88000],
    'credit_score': [720, 680, 750, 630, 700, 690, 710, None],
    'education':    ['Graduate', 'Graduate', 'Postgraduate', 'Undergraduate',
                     'Postgraduate', 'Graduate', 'Graduate', 'Undergraduate'],
    'employed':     [True, True, True, False, True, True, False, True],
    'loan_approved':[1, 1, 0, 0, 1, 1, 0, 1]
}

df = pd.DataFrame(data)
print('DataFrame:')
print(df)

DataFrame:
    age  income  credit_score      education  employed  loan_approved
0  25.0   50000         720.0       Graduate      True              1
1  34.0   82000         680.0       Graduate      True              1
2  45.0  120000         750.0   Postgraduate      True              0
3  28.0   45000         630.0  Undergraduate     False              0
4  52.0   95000         700.0   Postgraduate      True              1
5  31.0   60000         690.0       Graduate      True              1
6  38.0   74000         710.0       Graduate     False              0
7   NaN   88000           NaN  Undergraduate      True              1


## `info()` and `describe()` — Always Run These First

In [3]:
# info() tells you: column names, non-null counts, and data types
# Non-null count < total rows => missing values exist in that column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age            7 non-null      float64
 1   income         8 non-null      int64  
 2   credit_score   7 non-null      float64
 3   education      8 non-null      object 
 4   employed       8 non-null      bool   
 5   loan_approved  8 non-null      int64  
dtypes: bool(1), float64(2), int64(2), object(1)
memory usage: 460.0+ bytes


**Observation:** `age` and `credit_score` have 7 non-null values out of 8 rows — so there is one missing value in each. The `object` dtype for `education` means it is stored as strings. This is exactly what to look for at the start of any EDA.

In [4]:
# describe() gives statistics for numeric columns
df.describe().round(2)

,age,income,credit_score,loan_approved
count,7.00,8.00,7.00,8.00
mean,36.14,76750.00,697.14,0.62
std,9.62,24984.28,37.29,0.52
min,25.00,45000.00,630.00,0.00
25%,29.50,57500.00,685.00,0.00
50%,34.00,78000.00,700.00,1.00
75%,41.50,89750.00,715.00,1.00
max,52.00,120000.00,750.00,1.00


**Observation:** The `count` for `age` and `credit_score` is 7, confirming the missing values found in `info()`. The mean approval rate is 0.62, meaning 62% of applications were approved — a moderately unbalanced target variable.

In [5]:
# Quick missing value summary — a pattern used in every project
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)

missing_summary = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(missing_summary[missing_summary['missing_count'] > 0])

              missing_count  missing_%
age                       1       12.5
credit_score              1       12.5


## Selecting Data: `[]`, `.loc[]`, `.iloc[]`

In [6]:
# [] — select columns by name
print('Single column (Series):')
print(df['age'].values)

print('\nMultiple columns (DataFrame):')
print(df[['age', 'income', 'credit_score']].head(3))

Single column (Series):
[25. 34. 45. 28. 52. 31. 38. nan]

Multiple columns (DataFrame):
    age  income  credit_score
0  25.0   50000         720.0
1  34.0   82000         680.0
2  45.0  120000         750.0


In [7]:
# .loc[] — label-based: rows by index label, columns by name
# .iloc[] — position-based: rows and columns by integer position

print('.loc[2:4, "age":"education"]  (rows 2-4, columns age through education):')
print(df.loc[2:4, 'age':'education'])

print('\n.iloc[0:3, 0:3]  (first 3 rows, first 3 columns by position):')
print(df.iloc[0:3, 0:3])

.loc[2:4, "age":"education"]  (rows 2-4, columns age through education):
    age  income  credit_score      education
2  45.0  120000         750.0   Postgraduate
3  28.0   45000         630.0  Undergraduate
4  52.0   95000         700.0   Postgraduate

.iloc[0:3, 0:3]  (first 3 rows, first 3 columns by position):
    age  income  credit_score
0  25.0   50000         720.0
1  34.0   82000         680.0
2  45.0  120000         750.0


**Observation:** `.loc[]` includes the end label (row 4 is included), but `.iloc[]` excludes the end index (row 3 is not included). This is the same behaviour as Python slicing vs inclusive ranges.

## Filtering Rows

In [8]:
# Filter: approved applications with credit score above 700
mask = (df['loan_approved'] == 1) & (df['credit_score'] > 700)
approved_high_credit = df[mask]
print('Approved with credit score > 700:')
print(approved_high_credit[['age', 'income', 'credit_score', 'loan_approved']])

# .query() method — more readable for complex conditions
print('\nSame filter using .query():')
print(df.query('loan_approved == 1 and credit_score > 700')[['age', 'credit_score']])

Approved with credit score > 700:
    age  income  credit_score  loan_approved
0  25.0   50000         720.0              1

Same filter using .query():
    age  credit_score
0  25.0         720.0


## Handling Missing Values

In [9]:
# Strategy 1: Drop rows with any missing value
df_dropped = df.dropna()
print(f'Original: {len(df)} rows | After dropna: {len(df_dropped)} rows')

# Strategy 2: Fill numeric missing values with the column median
df_filled = df.copy()
for col in ['age', 'credit_score']:
    median_val = df_filled[col].median()
    df_filled[col].fillna(median_val, inplace=True)
    print(f'Filled "{col}" NaN with median = {median_val}')

print('\nMissing values after filling:')
print(df_filled.isnull().sum())

Original: 8 rows | After dropna: 7 rows
Filled "age" NaN with median = 34.0
Filled "credit_score" NaN with median = 700.0

Missing values after filling:
age              0
income           0
credit_score     0
education        0
employed         0
loan_approved    0
dtype: int64


C:\Users\mohit\AppData\Local\Temp\ipykernel_8816\3096937312.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filled[col].fillna(median_val, inplace=True)
C:\Users\mohit\AppData\Local\Temp\ipykernel_8816\3096937312.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

**Observation:** Dropping rows loses 2 out of 8 rows (25% of the dataset). In a small dataset like this, imputing with the median is clearly preferable. In larger datasets where missing values are under 5% of a column, either approach is reasonable.

## Adding and Removing Columns

In [10]:
df_work = df_filled.copy()

# Add a derived feature — income-to-age ratio
df_work['income_per_year_of_age'] = (df_work['income'] / df_work['age']).round(0)

# Add a categorical bin — age group
df_work['age_group'] = pd.cut(
    df_work['age'],
    bins=[0, 30, 40, 100],
    labels=['Under 30', '30-40', 'Over 40']
)

print(df_work[['age', 'income', 'income_per_year_of_age', 'age_group']].to_string())

# Remove a column
df_work.drop(columns=['income_per_year_of_age'], inplace=True)
print('\nColumns after drop:', df_work.columns.tolist())

    age  income  income_per_year_of_age age_group
0  25.0   50000                  2000.0  Under 30
1  34.0   82000                  2412.0     30-40
2  45.0  120000                  2667.0   Over 40
3  28.0   45000                  1607.0  Under 30
4  52.0   95000                  1827.0   Over 40
5  31.0   60000                  1935.0     30-40
6  38.0   74000                  1947.0     30-40
7  34.0   88000                  2588.0     30-40

Columns after drop: ['age', 'income', 'credit_score', 'education', 'employed', 'loan_approved', 'age_group']


## `groupby()` and `value_counts()`

In [11]:
# How does loan approval rate vary by education level?
approval_by_education = df_filled.groupby('education')['loan_approved'].agg(['mean', 'count'])
approval_by_education.columns = ['approval_rate', 'sample_count']
approval_by_education['approval_rate'] = approval_by_education['approval_rate'].round(2)
print('Loan approval by education:')
print(approval_by_education.sort_values('approval_rate', ascending=False))

# Distribution of education categories
print('\nEducation distribution:')
print(df_filled['education'].value_counts())

Loan approval by education:
               approval_rate  sample_count
education                                 
Graduate                0.75             4
Postgraduate            0.50             2
Undergraduate           0.50             2

Education distribution:
education
Graduate         4
Postgraduate     2
Undergraduate    2
Name: count, dtype: int64


**Observation:** Graduates have a 75% approval rate vs 50% for other education levels in this small dataset. With only 2 samples per group, this is not statistically reliable — it is an observation worth noting but not a conclusion to act on with this sample size.

## Sorting

In [12]:
# Sort by credit score descending, then by income ascending
sorted_df = df_filled.sort_values(
    by=['credit_score', 'income'],
    ascending=[False, True]
)
print(sorted_df[['age', 'income', 'credit_score', 'loan_approved']].to_string())

    age  income  credit_score  loan_approved
2  45.0  120000         750.0              0
0  25.0   50000         720.0              1
6  38.0   74000         710.0              0
7  34.0   88000         700.0              1
4  52.0   95000         700.0              1
5  31.0   60000         690.0              1
1  34.0   82000         680.0              1
3  28.0   45000         630.0              0


## Converting to NumPy — The Handoff to sklearn

In [13]:
# sklearn expects NumPy arrays (or at least array-like inputs)
# The typical pattern: select feature columns, separate target

feature_cols = ['age', 'income', 'credit_score']
target_col   = 'loan_approved'

X = df_filled[feature_cols].values   # .values converts to NumPy array
y = df_filled[target_col].values

print('X (features):')
print(X)
print('\ny (target):', y)
print('\nX dtype:', X.dtype, '| X shape:', X.shape)
print('y dtype:', y.dtype, '| y shape:', y.shape)

X (features):
[[2.5e+01 5.0e+04 7.2e+02]
 [3.4e+01 8.2e+04 6.8e+02]
 [4.5e+01 1.2e+05 7.5e+02]
 [2.8e+01 4.5e+04 6.3e+02]
 [5.2e+01 9.5e+04 7.0e+02]
 [3.1e+01 6.0e+04 6.9e+02]
 [3.8e+01 7.4e+04 7.1e+02]
 [3.4e+01 8.8e+04 7.0e+02]]

y (target): [1 1 0 0 1 1 0 1]

X dtype: float64 | X shape: (8, 3)
y dtype: int64 | y shape: (8,)


## Mini-Exercise: Data Summary Report Function

In [14]:
def dataset_report(df, target_col=None):
    """
    Print a concise summary of a DataFrame useful for ML preprocessing.
    Covers: shape, dtypes, missing values, numeric stats, and target distribution.

    Parameters:
        df         : pandas DataFrame
        target_col : optional — name of the target column for class distribution
    """
    print('=' * 55)
    print('DATASET REPORT')
    print('=' * 55)

    # Shape
    print(f'Shape          : {df.shape[0]} rows × {df.shape[1]} columns')

    # Column types summary
    type_counts = df.dtypes.value_counts()
    print('Column types   :', ', '.join(f'{v} {k}' for k, v in type_counts.items()))

    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print('Missing values : None')
    else:
        print('Missing values :')
        for col, count in missing.items():
            pct = count / len(df) * 100
            print(f'  {col:20s}: {count} ({pct:.1f}%)')

    # Numeric stats
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if target_col and target_col in numeric_cols:
        numeric_cols.remove(target_col)
    print(f'Numeric features: {numeric_cols}')

    # Categorical columns
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    print(f'Categorical cols: {cat_cols}')

    # Target distribution
    if target_col:
        print(f'\nTarget ({target_col}) distribution:')
        vc = df[target_col].value_counts()
        for val, count in vc.items():
            print(f'  {val}: {count} ({count/len(df)*100:.1f}%)')

    print('=' * 55)

dataset_report(df_filled, target_col='loan_approved')

DATASET REPORT
Shape          : 8 rows × 6 columns
Column types   : 2 float64, 2 int64, 1 object, 1 bool
Missing values : None
Numeric features: ['age', 'income', 'credit_score']
Categorical cols: ['education']

Target (loan_approved) distribution:
  1: 5 (62.5%)
  0: 3 (37.5%)


**Observation:** This function became a template used at the start of every mini-project in Weeks 3 and 4 to get an immediate overview before looking at anything else.

---

## Summary

| Operation | Pandas method | When to use |
|---|---|---|
| First look at types and nulls | `df.info()` | Always — run this first |
| Numeric statistics | `df.describe()` | Always — run this second |
| Select columns | `df[cols]` | Routine column selection |
| Select by label | `df.loc[rows, cols]` | When you know column names and index labels |
| Select by position | `df.iloc[rows, cols]` | When you know positions |
| Filter rows | `df[boolean_mask]` | Any row filtering |
| Missing values | `df.isnull().sum()`, `fillna()`, `dropna()` | Preprocessing step 1 |
| New features | `df['col'] = expr` | Feature engineering |
| Group analysis | `df.groupby(col).agg()` | Checking class-conditional statistics |
| To NumPy | `df[cols].values` | Before passing to sklearn |

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*